# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll list all record set `@id`s in the dataset and inspect their fields to guide downstream analysis.

In [ ]:
# List available record sets and their fields, referencing by `@id`
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")

for idx, rs in enumerate(record_sets):
    print(f"[{idx}] RecordSet @id: {rs['@id']}")
    print(f"    name: {rs.get('name', 'N/A')}")
    print("    Fields (by @id and name):")
    for field in rs.get('fields', []):
        print(f"      - @id: {field['@id']}, name: {field.get('name', 'unknown')}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

This example demonstrates extracting and displaying the first five records and columns for the main protein RecordSet.

In [ ]:
# Extract all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

# Load data for each record set into a DataFrame, keyed by @id
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading record set {record_set_id} ...")
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  - Columns: {dataframes[record_set_id].columns.tolist()}")
        print(f"  - # Rows: {len(dataframes[record_set_id])}\n")
    except Exception as e:
        print(f"  - Could not load record set {record_set_id}: {e}")
    print()
# For demonstration, select the primary record set (usually the first, but adjust if there's a specific main one)
main_record_set_id = record_set_ids[0]

print(f"Available columns in the main record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's perform common data processing tasks such as filtering, normalization, and grouping. All field references use their `@id` for precision.

We'll:
- Filter records for a numeric field (e.g. `coverage_percent`) above a threshold.
- Normalize this field.
- Group by a categorical attribute (e.g. `gene_symbol`).

In [ ]:
# Identify numeric and category fields by @id
# Replace these with actual field @ids from the overview if different
df = dataframes[main_record_set_id]
# Example guess: numeric_field might be '@id': 'coverage_percent', group_field '@id': 'gene_symbol'

# For demo, auto-detect a numeric column
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break

if numeric_field is None:
    # Try to convert columns to numeric where possible
    for col in df.columns:
        converted = pd.to_numeric(df[col], errors='coerce')
        if converted.notna().sum() > 0:
            df[col] = converted
            if numeric_field is None:
                numeric_field = col

print(f"Numeric field selected (by @id): {numeric_field}")
threshold = df[numeric_field].mean() if numeric_field else 0

filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (Count: {len(filtered_df)}):")
print(filtered_df.head())

# Normalize
filtered_df = filtered_df.copy()  # avoid SettingWithCopyWarning
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()

print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by a field likely categorical:
possible_group_fields = [col for col in df.columns if col != numeric_field and df[col].nunique() < len(df) // 2 and df[col].dtype==object]
group_field = possible_group_fields[0] if possible_group_fields else None
if group_field:
    print(f"Grouping by field (by @id): {group_field}\n")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and compare means by group if a grouping field was found.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f'Distribution of field: {numeric_field}')
plt.xlabel(numeric_field)
plt.show()

# If grouping field exists, plot group means
if group_field:
    mean_by_group = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
    plt.figure(figsize=(10,5))
    sns.barplot(y=mean_by_group.index, x=mean_by_group.values)
    plt.title(f'Top 10 {group_field} mean {numeric_field}')
    plt.xlabel(f'Mean {numeric_field}')
    plt.ylabel(group_field)
    plt.show()

## 6. Conclusion

This notebook demonstrated how to use the `mlcroissant` library to explore the FAIR² dataset defined by the Croissant schema.

- We identified the dataset's record sets and fields by their `@id` values.
- Loaded and previewed the records in pandas DataFrames.
- Performed basic filtering, normalization, and grouping—referencing all fields by their stable `@id` values.
- Visualized the distribution of a numeric variable as well as group means for exploratory analysis.

For further analysis, refer to the documentation of [mlcroissant](https://pypi.org/project/mlcroissant/) and the Croissant dataset for domain-specific field identifiers and metadata.